# Notebook 04 - MONAI Brain Tumor Segmentation

**Project:** Explainable Deep Learning for MRI Brain Tumor Classification and Segmentation Using Transfer Learning, MONAI, U-Net, and Grad-CAM

Run notebooks in order. Each notebook writes outputs into the same project folder so later notebooks can reuse them.


## Purpose

This notebook trains a **MONAI 3D segmentation model** on BraTS-style multimodal MRI.

Input modalities:

- T1
- T1ce / T1c
- T2
- FLAIR

Output channels:

- Tumor core (TC)
- Whole tumor (WT)
- Enhancing tumor (ET)

Start with a small subset in Colab, then increase cases/epochs when the pipeline works.


In [ ]:
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "monai[nibabel]", "nibabel", "torch", "numpy",
                           "pandas", "matplotlib", "tqdm"])
print("Running in Colab:", IN_COLAB)


In [ ]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch.utils.data import DataLoader
import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped, Orientationd,
    NormalizeIntensityd, CropForegroundd, RandSpatialCropd, RandFlipd,
    RandRotate90d, RandScaleIntensityd, RandShiftIntensityd, Activations,
    AsDiscrete, MapTransform
)
from monai.data import CacheDataset, decollate_batch
from monai.networks.nets import UNet, SegResNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric, MeanIoU
from monai.inferers import sliding_window_inference

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("MONAI:", monai.__version__)
print("Device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
if IN_COLAB:
    PROJECT_ROOT = Path("/content/brain_tumor_xai_project")
else:
    PROJECT_ROOT = Path.cwd() / "brain_tumor_xai_project"

config_path = PROJECT_ROOT / "project_config.json"
if config_path.exists():
    config = json.loads(config_path.read_text())
    PROJECT_ROOT = Path(config["project_root"])
    MODEL_DIR = Path(config["model_dir"])
    FIGURE_DIR = Path(config["figure_dir"])
else:
    MODEL_DIR = PROJECT_ROOT / "models"
    FIGURE_DIR = PROJECT_ROOT / "figures"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
SEG_FIGURE_DIR = FIGURE_DIR / "segmentation"
for p in [OUTPUT_DIR, MODEL_DIR, SEG_FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# CHANGE THIS to your BraTS folder.
BRATS_ROOT = PROJECT_ROOT / "data" / "brats_training_data"
print("BRATS_ROOT:", BRATS_ROOT)


In [ ]:
def find_file(case_dir, patterns):
    for pat in patterns:
        matches = list(case_dir.glob(pat))
        if matches:
            return str(matches[0])
    return None

def build_brats_datalist(root, max_cases=None):
    root = Path(root)
    case_dirs = sorted([p for p in root.iterdir() if p.is_dir()]) if root.exists() else []
    data = []
    for case_dir in case_dirs:
        flair = find_file(case_dir, ["*flair*.nii.gz", "*FLAIR*.nii.gz"])
        t1 = find_file(case_dir, ["*t1.nii.gz", "*T1.nii.gz", "*_t1_*.nii.gz"])
        t1ce = find_file(case_dir, ["*t1ce*.nii.gz", "*t1c*.nii.gz", "*T1CE*.nii.gz"])
        t2 = find_file(case_dir, ["*t2.nii.gz", "*T2.nii.gz", "*_t2_*.nii.gz"])
        seg = find_file(case_dir, ["*seg*.nii.gz", "*mask*.nii.gz", "*label*.nii.gz"])
        if all([t1, t1ce, t2, flair, seg]):
            data.append({"image": [t1, t1ce, t2, flair], "label": seg, "case_id": case_dir.name})
    return data[:max_cases] if max_cases else data

MAX_CASES = 20
all_data = build_brats_datalist(BRATS_ROOT, MAX_CASES)
if not all_data:
    raise FileNotFoundError("No BraTS cases found. Set BRATS_ROOT to your BraTS-style dataset folder.")
print("Cases found:", len(all_data))
print(all_data[0])


In [ ]:
random.shuffle(all_data)
n_val = max(1, int(len(all_data) * 0.2))
train_files = all_data[:-n_val]
val_files = all_data[-n_val:]
print("Train cases:", len(train_files), "Val cases:", len(val_files))
(OUTPUT_DIR / "brats_datalist.json").write_text(json.dumps({"train": train_files, "val": val_files}, indent=2))


In [ ]:
class ConvertToMultiChannelBasedOnBratsClassesd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            label = d[key]
            if label.shape[0] == 1:
                label = label[0]
            tc = torch.logical_or(label == 1, label == 4)
            wt = torch.logical_or(torch.logical_or(label == 1, label == 2), label == 4)
            et = label == 4
            d[key] = torch.stack([tc, wt, et], dim=0).float()
        return d


In [ ]:
ROI_SIZE = (96, 96, 96)
BATCH_SIZE = 1
NUM_WORKERS = 2 if IN_COLAB else 0
CACHE_RATE = 0.2

train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image"]),
    EnsureChannelFirstd(keys=["label"], channel_dim="no_channel"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys=["label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandSpatialCropd(keys=["image", "label"], roi_size=ROI_SIZE, random_size=False),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5),
    RandShiftIntensityd(keys=["image"], offsets=0.1, prob=0.5),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image"]),
    EnsureChannelFirstd(keys=["label"], channel_dim="no_channel"),
    EnsureTyped(keys=["image", "label"]),
    ConvertToMultiChannelBasedOnBratsClassesd(keys=["label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
])

train_ds = CacheDataset(train_files, train_transforms, cache_rate=CACHE_RATE, num_workers=NUM_WORKERS)
val_ds = CacheDataset(val_files, val_transforms, cache_rate=CACHE_RATE, num_workers=NUM_WORKERS)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

b = next(iter(train_loader))
print("Image shape:", b["image"].shape)
print("Label shape:", b["label"].shape)


In [ ]:
SEG_MODEL_NAME = "unet"  # change to "segresnet" if resources allow

if SEG_MODEL_NAME == "unet":
    model = UNet(spatial_dims=3, in_channels=4, out_channels=3,
                 channels=(16, 32, 64, 128), strides=(2, 2, 2), num_res_units=2)
elif SEG_MODEL_NAME == "segresnet":
    model = SegResNet(spatial_dims=3, init_filters=16, in_channels=4, out_channels=3, dropout_prob=0.2)
else:
    raise ValueError(SEG_MODEL_NAME)

model = model.to(device)
loss_function = DiceCELoss(sigmoid=True, squared_pred=True, smooth_nr=1e-5, smooth_dr=1e-5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
dice_metric = DiceMetric(include_background=True, reduction="mean")
iou_metric = MeanIoU(include_background=True, reduction="mean")
post_pred = Compose([Activations(sigmoid=True), AsDiscrete(threshold=0.5)])
post_label = Compose([AsDiscrete(threshold=0.5)])
print(model.__class__.__name__)


In [ ]:
MAX_EPOCHS = 5  # final run can be 30-100 depending on GPU time
VAL_INTERVAL = 1
best_metric, best_epoch = -1, -1
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss, steps = 0, 0
    for batch_data in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS}"):
        steps += 1
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= max(steps, 1)

    row = {"epoch": epoch, "train_loss": epoch_loss}
    print("Train loss:", epoch_loss)

    if epoch % VAL_INTERVAL == 0:
        model.eval()
        dice_metric.reset(); iou_metric.reset()
        with torch.no_grad():
            for val_data in tqdm(val_loader, desc="Validation"):
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)
                val_outputs = sliding_window_inference(val_inputs, ROI_SIZE, 1, model)
                val_outputs = [post_pred(x) for x in decollate_batch(val_outputs)]
                val_labels = [post_label(x) for x in decollate_batch(val_labels)]
                dice_metric(y_pred=val_outputs, y=val_labels)
                iou_metric(y_pred=val_outputs, y=val_labels)
        mean_dice = float(dice_metric.aggregate().item())
        mean_iou = float(iou_metric.aggregate().item())
        row["val_dice"] = mean_dice
        row["val_iou"] = mean_iou
        print("Val Dice:", mean_dice, "Val IoU:", mean_iou)

        if mean_dice > best_metric:
            best_metric = mean_dice
            best_epoch = epoch
            torch.save({"model_name": SEG_MODEL_NAME, "state_dict": model.state_dict(),
                        "roi_size": ROI_SIZE, "best_dice": best_metric, "epoch": epoch},
                       MODEL_DIR / f"{SEG_MODEL_NAME}_segmentation_best.pt")
    history.append(row)

hist_df = pd.DataFrame(history)
hist_df.to_csv(OUTPUT_DIR / f"{SEG_MODEL_NAME}_segmentation_history.csv", index=False)
print("Best Dice:", best_metric, "Epoch:", best_epoch)
hist_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hist_df["epoch"], hist_df["train_loss"], marker="o")
plt.title(f"{SEG_MODEL_NAME} Training Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.tight_layout()
plt.savefig(SEG_FIGURE_DIR / f"{SEG_MODEL_NAME}_segmentation_loss_curve.png", dpi=200)
plt.show()

if "val_dice" in hist_df:
    plt.figure(figsize=(8, 5))
    plt.plot(hist_df["epoch"], hist_df["val_dice"], marker="o", label="Dice")
    plt.plot(hist_df["epoch"], hist_df["val_iou"], marker="o", label="IoU")
    plt.title(f"{SEG_MODEL_NAME} Validation Metrics")
    plt.xlabel("Epoch"); plt.ylabel("Score"); plt.legend(); plt.tight_layout()
    plt.savefig(SEG_FIGURE_DIR / f"{SEG_MODEL_NAME}_segmentation_metrics_curve.png", dpi=200)
    plt.show()


In [ ]:
ckpt_path = MODEL_DIR / f"{SEG_MODEL_NAME}_segmentation_best.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
model.eval()

sample = val_ds[0]
image = sample["image"].unsqueeze(0).to(device)
label = sample["label"].to(device)

with torch.no_grad():
    pred = sliding_window_inference(image, ROI_SIZE, 1, model)
    pred = torch.sigmoid(pred)[0].cpu().numpy()

image_np = sample["image"].cpu().numpy()
label_np = label.cpu().numpy()
pred_bin = (pred > 0.5).astype(np.float32)

channel = 1  # whole tumor
slice_sums = label_np[channel].sum(axis=(0, 1))
z = int(np.argmax(slice_sums)) if slice_sums.max() > 0 else image_np.shape[-1] // 2

flair_slice = image_np[3, :, :, z]
gt_slice = label_np[channel, :, :, z]
pred_slice = pred_bin[channel, :, :, z]

plt.figure(figsize=(14, 4))
plt.subplot(1, 4, 1); plt.imshow(flair_slice, cmap="gray"); plt.title("FLAIR MRI"); plt.axis("off")
plt.subplot(1, 4, 2); plt.imshow(gt_slice, cmap="gray"); plt.title("Ground truth WT"); plt.axis("off")
plt.subplot(1, 4, 3); plt.imshow(pred_slice, cmap="gray"); plt.title("Predicted WT"); plt.axis("off")
plt.subplot(1, 4, 4); plt.imshow(flair_slice, cmap="gray"); plt.imshow(pred_slice, alpha=0.4); plt.title("Prediction overlay"); plt.axis("off")
plt.tight_layout()
fig_path = SEG_FIGURE_DIR / f"{SEG_MODEL_NAME}_segmentation_prediction_example.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved:", fig_path)


In [ ]:
summary = {
    "model": SEG_MODEL_NAME,
    "max_epochs": MAX_EPOCHS,
    "roi_size": str(ROI_SIZE),
    "best_val_dice": best_metric,
    "best_metric_epoch": best_epoch,
    "train_cases": len(train_files),
    "val_cases": len(val_files),
    "batch_size": BATCH_SIZE,
}
(OUTPUT_DIR / "segmentation_summary.json").write_text(json.dumps(summary, indent=2))
summary


## Outputs from Notebook 04

- `*_segmentation_best.pt`
- segmentation history CSV
- Dice and IoU values
- segmentation mask overlay figure
- `segmentation_summary.json`

Next: run `05_results_analysis_and_paper_assets.ipynb`.
